In [ ]:
# Quarterly depedency graph

In [ ]:
import os
os.getcwd()

In [ ]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [ ]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [ ]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [ ]:
entsog

In [ ]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [ ]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [ ]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [ ]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [ ]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [ ]:
# to align with Bloombgerg lng
# agsi_19= agsi_19.iloc[:0] #Ugne change this one!

In [ ]:
agsi_19

In [ ]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [ ]:
agsi_19
#agsi_19= agsi_19.iloc[:-1]

In [ ]:
#ratios_df= ratios_df.iloc[:-1] #Ugne change this one!

In [ ]:
ratios_df

In [ ]:
#lng = lng.iloc[:-1]

In [ ]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [ ]:
lng

In [ ]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [ ]:
lng['Total less Russia and USA and NO and AL'] = lng['Tot'] - lng['Russia'] - lng['United States'] - lng['Norway'] - lng['Algeria']

In [ ]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2026-03-31', freq='M')

In [ ]:
entsog = entsog['2019':]

In [ ]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [ ]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [ ]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [ ]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [ ]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
graph['Norway LNG'] = lng_q['Norway']
graph['Algeria LNG'] = lng_q['Algeria']
del graph['Russia']
graph['LNG less RU and USA and NO and AL'] = lng_q['Total less Russia and USA and NO and AL']
graph = graph['2019':]

In [ ]:
graph.tail()

In [ ]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'Norway LNG', 'USA LNG', 'Algeria LNG', 'LNG less RU and USA and NO and AL', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [ ]:
from datetime import datetime
today = date.today()

In [ ]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()